# Micro-TTT Edge: Google Colab Runner
**Project:** Single-step rank-1 closed-form test-time memory in CPU L2 cache
**Author:** Jaime Jarauta
**Pre-Registered Question:** Can closed-form rank-1 outer-product test-time updates achieve streaming recall on CPU at > 100 tok/s within < 16KB state?

In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Unassigning Colab GPU immediately to prevent idle burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# Step 2: Ingest Codebase & Dependencies
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("micro-ttt-edge"):
        os.system("git clone -q -b run https://github.com/jaimejarauta/micro-ttt-edge.git")
    if "micro-ttt-edge/src" not in sys.path:
        sys.path.insert(0, "micro-ttt-edge/src")

if "src" not in sys.path:
    sys.path.insert(0, "src")

try:
    from micro_ttt.model import MicroTTTModel
    from micro_ttt.tasks import generate_passkey_retrieval_batch
    from micro_ttt.metrics import benchmark_cpu_throughput
    print("[Ready] Loaded Micro-TTT components.")
except Exception as e:
    print(f"Import error: {e}")


In [ ]:
# Step 3: Run Smoke Benchmark
model = MicroTTTModel(vocab_size=256, d_model=64, d_k=32, d_v=32, num_layers=2).to(device)
metrics = benchmark_cpu_throughput(model, seq_len=2048, batch_size=2, device=device)
print("=" * 60)
print(f"Streaming Throughput : {metrics['tokens_per_sec']:.1f} tok/s (Target: >= 100 tok/s)")
print(f"Per-Token Latency    : {metrics['ms_per_token']:.3f} ms/token")
print(f"Inner State Footprint: {metrics['state_kb']:.1f} KB (Target: <= 32 KB)")
gate_passed = metrics['tokens_per_sec'] >= 100 and metrics['state_kb'] <= 32
print(f"Pre-Registered Gate 1: {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 4: Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}
if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning runtime immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Skipped auto-kill: {e})")
